#Task 2 Approach 1
# Using BiLSTM-CRF approach

In [ ]:
#!pip uninstall -y scipy numpy gensim
#!pip install scipy numpy gensim --no-cache-dir --force-reinstall
#!python -m nltk.downloader punkt
#!python -m spacy download en_core_web_sm
#!pip install seqeval
#!pip install pytorch-crf

In [ ]:
import pandas as pd
import gc

# Mount Google Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# File paths
train_path = "/content/drive/My Drive/ANLE/Assignment Datasets/propaganda_train.tsv"
val_path = "/content/drive/My Drive/ANLE/Assignment Datasets/propaganda_val.tsv"

# Read the TSV files into DataFrames
train_df = pd.read_csv(train_path, sep='\t')
val_df = pd.read_csv(val_path, sep='\t')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train_df.head()

,label,tagged_in_context
0,not_propaganda,"No, <BOS> he <EOS> will not be confirmed."
1,not_propaganda,This declassification effort <BOS> won’t make ...
2,flag_waving,The Obama administration misled the <BOS> Amer...
3,not_propaganda,“It looks like we’re capturing the demise of t...
4,not_propaganda,"<BOS> Location: Westerville, Ohio <EOS>"


#### Resample non-propaganda instances

In [ ]:
print(train_df.shape)
print(val_df.shape)

(2414, 2)
(580, 2)


#### Downsample data sets

In [ ]:
import numpy as np

def resample_non_propaganda(df, keep_prob=0.8, seed=42):
    """
    Randomly drops non-propaganda instances from the DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame with 'label' column.
        keep_prob (float): Probability of keeping a non-propaganda instance.
        seed (int): Random seed for reproducibility.

    Returns:
        pd.DataFrame: A resampled DataFrame.
    """
    np.random.seed(seed)

    # Split into propaganda and non-propaganda
    is_non_propaganda = df['label'].str.strip() == "not_propaganda"

    # Keep all propaganda instances
    propaganda_df = df[~is_non_propaganda]

    # Randomly keep some non-propaganda instances
    non_propaganda_df = df[is_non_propaganda]
    keep_mask = np.random.rand(len(non_propaganda_df)) < keep_prob
    kept_non_propaganda = non_propaganda_df[keep_mask]

    # Combine back
    resampled_df = pd.concat([propaganda_df, kept_non_propaganda]).reset_index(drop=True)

    return resampled_df

train_df = resample_non_propaganda(train_df, keep_prob=0.3)
val_df = resample_non_propaganda(val_df, keep_prob=0.3)

In [ ]:
print(train_df.shape)
print(val_df.shape)

(1597, 2)
(375, 2)


## Preprocess Data and generate BIO tags

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')

def extract_tokens_and_bio_tags(row):
    sentence = row['tagged_in_context']
    label = row['label'].strip()

    # Extract span between <BOS> and <EOS>
    span_match = re.search(r"<BOS>(.*?)<EOS>", sentence)
    if not span_match:
        # No span found, return all O's
        tokens = word_tokenize(sentence.replace("<BOS>", "").replace("<EOS>", ""))
        return tokens, ['O'] * len(tokens)

    # Extract and clean span
    span = span_match.group(1).strip()
    span_tokens = word_tokenize(span)

    # Tokenize full sentence (without <BOS>/<EOS>)
    clean_sentence = sentence.replace("<BOS>", "").replace("<EOS>", "")
    tokens = word_tokenize(clean_sentence)

    # Initialize all tags to O
    tags = ['O'] * len(tokens)

    # Find the matching span indices
    for i in range(len(tokens) - len(span_tokens) + 1):
        if tokens[i:i+len(span_tokens)] == span_tokens:
            tags[i] = f"B-{label}"
            for j in range(1, len(span_tokens)):
                tags[i+j] = f"I-{label}"
            break  # Stop after first match

    return tokens, tags

# Remove tags as they are not useful for this approach
def clean_text(text):
    return text.replace('<BOS>', '').replace('<EOS>', '').strip()


# Convert all non-propaganda sents to fully not_propaganda tags (for experiment)
def reassign_non_propaganda_spans(row):
    tokens = row['tokenised_sents']
    label = row['label'].strip()

    if label == "not_propaganda":
        # Full sentence as one span
        tags = ['B-not_propaganda'] + ['I-not_propaganda'] * (len(tokens) - 1)
        return tokens, tags
    else:
        # Keep original spans for propaganda
        return row['tokenised_sents'], row['BIO_tags']

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# Apply the function to both train and val DataFrames
train_tokens_and_tags = train_df.apply(extract_tokens_and_bio_tags, axis=1)
val_tokens_and_tags = val_df.apply(extract_tokens_and_bio_tags, axis=1)

# Split tuple columns into two new columns
train_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(train_tokens_and_tags.tolist(), index=train_df.index)
val_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(val_tokens_and_tags.tolist(), index=val_df.index)

# Apply the function to reformat all non-propaganda tags
train_tokens_and_tags = train_df.apply(reassign_non_propaganda_spans, axis=1)
val_tokens_and_tags = val_df.apply(reassign_non_propaganda_spans, axis=1)

train_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(train_tokens_and_tags.tolist(), index=train_df.index)
val_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(val_tokens_and_tags.tolist(), index=val_df.index)

train_df['sents'] = train_df['tagged_in_context'].apply(clean_text)

In [ ]:
# Rebuild train_df
train_df = pd.DataFrame({
    'sents': train_df['tagged_in_context'].apply(clean_text),
    'tokenised_sents': train_df['tokenised_sents'],
    'BIO_tags': train_df['BIO_tags']
})

# Same for val_df
val_df = pd.DataFrame({
    'sents': val_df['tagged_in_context'].apply(clean_text),
    'tokenised_sents': val_df['tokenised_sents'],
    'BIO_tags': val_df['BIO_tags']
})

In [ ]:
train_df.head(50)

,sents,tokenised_sents,BIO_tags
0,The Obama administration misled the American ...,"[The, Obama, administration, misled, the, Amer...","[O, O, O, O, O, B-flag_waving, I-flag_waving, ..."
1,"Hitler annihilated 400,000 Germans who were ...","[Hitler, annihilated, 400,000, Germans, who, w...","[O, B-loaded_language, O, O, O, O, O, O, O, O,..."
2,"As noted above, at this point literally every ...","[As, noted, above, ,, at, this, point, literal...","[O, O, O, O, O, O, O, O, O, O, O, B-doubt, I-d..."
3,His account was suspended for violating Twitte...,"[His, account, was, suspended, for, violating,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-name..."
4,A couple of seemingly unrelated events this pa...,"[A, couple, of, seemingly, unrelated, events, ...","[O, O, O, O, O, O, O, O, O, B-appeal_to_fear_p..."
5,Trump has expelled more Russian diplomats an...,"[Trump, has, expelled, more, Russian, diplomat...","[O, O, O, B-repetition, O, O, O, O, O, O, O, O..."
6,"But who authorized this strategic commitment ,...","[But, who, authorized, this, strategic, commit...","[B-doubt, I-doubt, I-doubt, I-doubt, I-doubt, ..."
7,"Outside of specialists and seekers, the only ...","[Outside, of, specialists, and, seekers, ,, th...","[O, O, O, O, O, O, B-causal_oversimplification..."
8,We must ask ourselves why this hasn’t happened .,"[We, must, ask, ourselves, why, this, hasn, ’,...","[B-doubt, I-doubt, I-doubt, I-doubt, I-doubt, ..."
9,We’ve caught on to his sneaky synodal tactics.,"[We, ’, ve, caught, on, to, his, sneaky, synod...","[O, O, O, O, O, O, O, B-loaded_language, O, O, O]"


### Create Label Mappings and Convert BIO Tags to ID Sequences

In [ ]:
# Collect all unique BIO tags from train and val
all_tags = set(tag for tags in train_df['BIO_tags'].tolist() + val_df['BIO_tags'].tolist() for tag in tags)

# Ensure "O" comes first
all_tags.discard('O')  # Remove 'O' temporarily
all_tags = sorted(all_tags)  # Sort all other tags alphabetically
all_tags = ['O'] + all_tags   # Add 'O' at the beginning

# Create mapping dictionaries
label2id = {label: idx for idx, label in enumerate(all_tags)}
id2label = {idx: label for label, idx in label2id.items()}

# Encode labels in DataFrame
train_df['label_ids'] = train_df['BIO_tags'].apply(lambda tags: [label2id[tag] for tag in tags])
val_df['label_ids'] = val_df['BIO_tags'].apply(lambda tags: [label2id[tag] for tag in tags])

print(label2id)

{'O': 0, 'B-appeal_to_fear_prejudice': 1, 'B-causal_oversimplification': 2, 'B-doubt': 3, 'B-exaggeration,minimisation': 4, 'B-flag_waving': 5, 'B-loaded_language': 6, 'B-name_calling,labeling': 7, 'B-not_propaganda': 8, 'B-repetition': 9, 'I-appeal_to_fear_prejudice': 10, 'I-causal_oversimplification': 11, 'I-doubt': 12, 'I-exaggeration,minimisation': 13, 'I-flag_waving': 14, 'I-loaded_language': 15, 'I-name_calling,labeling': 16, 'I-not_propaganda': 17, 'I-repetition': 18}


In [ ]:
train_df.head()

,sents,tokenised_sents,BIO_tags,label_ids
0,The Obama administration misled the American ...,"[The, Obama, administration, misled, the, Amer...","[O, O, O, O, O, B-flag_waving, I-flag_waving, ...","[0, 0, 0, 0, 0, 5, 14, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,"Hitler annihilated 400,000 Germans who were ...","[Hitler, annihilated, 400,000, Germans, who, w...","[O, B-loaded_language, O, O, O, O, O, O, O, O,...","[0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,"As noted above, at this point literally every ...","[As, noted, above, ,, at, this, point, literal...","[O, O, O, O, O, O, O, O, O, O, O, B-doubt, I-d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 12, 0, 0,..."
3,His account was suspended for violating Twitte...,"[His, account, was, suspended, for, violating,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-name...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 16,..."
4,A couple of seemingly unrelated events this pa...,"[A, couple, of, seemingly, unrelated, events, ...","[O, O, O, O, O, O, O, O, O, B-appeal_to_fear_p...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 10, 10, 10, 10,..."


## Generate Features (Word2Vec + PoS + NEs + TF-IDF)

In [ ]:
import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import KeyedVectors
from nltk.tokenize import word_tokenize
from tqdm import tqdm

tqdm.pandas()

# Load spaCy and Word2Vec
nlp = spacy.load("en_core_web_sm")
w2v = KeyedVectors.load_word2vec_format('/content/drive/My Drive/ANLE/Embeddings/GoogleNews-vectors-negative300.bin.gz', binary=True)

In [ ]:
# Prepare TF-IDF on full corpus (train + val tokenized sentences as strings)

# Combine tokenized sentences from train and val
all_tokenized = pd.concat([
    train_df['tokenised_sents'],
    val_df['tokenised_sents']
])

# Turn list of tokens into strings
all_sentences = all_tokenized.apply(lambda tokens: ' '.join(tokens)).tolist()

# Fit TF-IDF vectorizer on the full dataset
vectorizer = TfidfVectorizer()
vectorizer.fit(all_sentences)

# Get IDF scores for each token
idf_dict = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))

# Get min, max values for normalisation
idf_values = np.array(list(idf_dict.values()))
idf_min, idf_max = idf_values.min(), idf_values.max()

# Function for normalisation
def normalize_idf(score):
    return (score - idf_min) / (idf_max - idf_min)

In [ ]:
def one_hot(value, allowed_values):
    vec = np.zeros(len(allowed_values), dtype=np.float32)
    if value in allowed_values:
        vec[allowed_values.index(value)] = 1.0
    return vec

all_pos_tags = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ',
                'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT',
                'SCONJ', 'SYM', 'VERB', 'X', 'SPACE']

all_ner_labels = ['PERSON', 'ORG', 'GPE', 'LOC', 'DATE', 'TIME',
                  'MONEY', 'QUANTITY', 'ORDINAL', 'CARDINAL', 'NORP', 'LAW', 'FAC', 'EVENT', 'WORK_OF_ART', 'LANGUAGE']

def extract_features(tokens):
    doc = nlp(' '.join(tokens))
    features = []

    for i, token in enumerate(tokens):
        # Word2Vec embedding
        if token in w2v:
            w2v_vec = w2v[token]
        else:
            w2v_vec = np.zeros(w2v.vector_size)

        # TF-IDF score (normalized to a scalar feature)
        raw_score = idf_dict.get(token.lower(), 0.0)
        tfidf_score = normalize_idf(raw_score)
        tfidf_vec = np.array([tfidf_score])

        # POS tag
        pos_tag = doc[i].pos_
        pos_vec = one_hot(pos_tag, all_pos_tags)

        # NER tag
        ner_tag = doc[i].ent_type_
        ner_vec = one_hot(ner_tag, all_ner_labels)

        # Combine features
        combined = np.concatenate([w2v_vec, tfidf_vec, pos_vec, ner_vec])
        features.append(combined)

    return np.array(features)

In [ ]:
train_df['features'] = train_df['tokenised_sents'].progress_apply(extract_features)
val_df['features'] = val_df['tokenised_sents'].progress_apply(extract_features)

100%|██████████| 375/375 [00:03<00:00, 110.86it/s]


In [ ]:
train_df['features'][8].shape

(11, 335)

In [ ]:
# Free up some RAM
del w2v

## Generate Paddings for the train and val datasets

In [ ]:
def pad_instance(features, labels, max_len, feature_dim, pad_label=0):
    """
    Pads a single instance (features + labels) to max_len
    """
    seq_len = features.shape[0]

    # Pad features
    padded_feats = np.zeros((max_len, feature_dim), dtype=np.float32)
    padded_feats[:seq_len] = features

    # Pad labels
    padded_labels = np.full((max_len,), fill_value=pad_label, dtype=np.int64)
    padded_labels[:seq_len] = labels

    # Attention mask where 1 for real token, 0 for padding
    attention_mask = np.zeros((max_len,), dtype=np.uint8)
    attention_mask[:seq_len] = 1

    return padded_feats, padded_labels, attention_mask

def pad_dataset(df, pad_label=0):
    """
    Pads sequences of features and labels in a dataset to a uniform length.
    """

    # Determine the dimensionality of the input features (number of features per time step)
    feature_dim = df['features'].iloc[0].shape[1]

    # Find the longest sequence length in the dataset to determine how much to pad
    max_len = max(len(feats) for feats in df['features'])

    # Lists to collect padded outputs
    padded_features = []     # Will store padded 3D feature arrays
    padded_labels = []       # Will store padded label sequences
    attention_masks = []     # Will store attention masks (1 for real data, 0 for padding)

    # Process each example in the dataset
    for feats, labels in zip(df['features'], df['label_ids']):
        # Convert to NumPy arrays for padding function
        pf, pl, mask = pad_instance(
            features=np.array(feats),    # Shape: (seq_len, feature_dim)
            labels=np.array(labels),     # Shape: (seq_len,)
            max_len=max_len,             # Target sequence length after padding
            feature_dim=feature_dim,     # Number of features per time step
            pad_label=pad_label          # Label to use for padding label sequence
        )

        # Store padded outputs
        padded_features.append(pf)       # Shape: (max_len, feature_dim)
        padded_labels.append(pl)         # Shape: (max_len,)
        attention_masks.append(mask)     # Shape: (max_len,)

    # Convert lists to NumPy arrays with uniform shapes
    return (
        np.stack(padded_features),       # (num_examples, max_len, feature_dim)
        np.stack(padded_labels),         # (num_examples, max_len)
        np.stack(attention_masks)        # (num_examples, max_len)
    )

In [ ]:
all_label_ids = [lid for seq in train_df['label_ids'] for lid in seq]
print("Min label ID:", min(all_label_ids))
print("Max label ID:", max(all_label_ids))
print("Number of unique labels:", len(set(all_label_ids)))
print("num_labels in model:", len(label2id))

Min label ID: 0
Max label ID: 18
Number of unique labels: 19
num_labels in model: 19


#### Analyse label imbalancement

In [ ]:
from collections import Counter
all_labels = [label for seq in train_df['BIO_tags'] for label in seq]
c= Counter(all_labels)

print(c)

Counter({'O': 30532, 'I-not_propaganda': 9537, 'I-causal_oversimplification': 3591, 'I-doubt': 3166, 'I-appeal_to_fear_prejudice': 2722, 'I-flag_waving': 1599, 'I-exaggeration,minimisation': 1155, 'I-name_calling,labeling': 590, 'I-loaded_language': 413, 'B-not_propaganda': 374, 'I-repetition': 305, 'B-exaggeration,minimisation': 163, 'B-causal_oversimplification': 158, 'B-name_calling,labeling': 157, 'B-loaded_language': 152, 'B-appeal_to_fear_prejudice': 150, 'B-flag_waving': 147, 'B-repetition': 146, 'B-doubt': 144})


## Wrap the Data in a PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class PropagandaDataset(Dataset):
    def __init__(self, features, labels, masks):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.masks = torch.tensor(masks, dtype=torch.bool)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {
            'features': self.features[idx],
            'labels': self.labels[idx],
            'mask': self.masks[idx]
        }

In [ ]:
del all_tags, nlp, all_sentences, vectorizer, idf_dict, idf_values
gc.collect()

54

#### I generate paddings here to use them straight away and delete data to avoid exessive RAM and crashes. Seems messy, but the only way for notebook not to crash

In [ ]:
# Function to extarct only propaganda instance (for experiment)
def propaganda_only(tags):
    return any(tag != "O" and "not_propaganda" not in tag for tag in tags)

train_df_prop_only = train_df[train_df['BIO_tags'].apply(propaganda_only)].reset_index(drop=True)

span_feats, span_labels, span_masks = pad_dataset(train_df_prop_only)
span_dataset = PropagandaDataset(span_feats,span_labels, span_masks)
del train_df_spans_only

In [ ]:
val_feats, val_labels, val_masks = pad_dataset(val_df)
val_dataset = PropagandaDataset(val_feats, val_labels, val_masks)
del val_feats, val_labels, val_masks


In [ ]:
train_feats, train_labels, train_masks = pad_dataset(train_df)
train_dataset = PropagandaDataset(train_feats, train_labels, train_masks)
del train_feats, train_labels, train_masks, train_df, val_df

In [ ]:
print(f"Span Only Data Features: {span_feats.shape}, Labels: {span_labels.shape}, Masks: {span_masks.shape}")
print(f"Train Data Features:     {train_feats.shape}, Labels: {train_labels.shape}, Masks: {train_masks.shape}")
print(f"Val Data Features:       {val_feats.shape}, Labels: {val_labels.shape}, Masks: {val_masks.shape}")

Span Only Data Features: (1217, 1365, 335), Labels: (1217, 1365), Masks: (1217, 1365)
Train Data Features:     (1597, 1365, 335), Labels: (1597, 1365), Masks: (1597, 1365)
Val Data Features:       (375, 502, 335), Labels: (375, 502), Masks: (375, 502)


## Create DataLoaders for Training

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
del train_dataset
val_loader = DataLoader(val_dataset, batch_size=16)
del val_dataset
prop_loader = DataLoader(span_dataset, batch_size=16, shuffle=True)

## BiLSTM-CRF Model

This code was based on the code available at: https://github.com/LiyuanLucasLiu/LM-LSTM-CRF/tree/master/model

In [ ]:
import torch
import torch.nn as nn
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_labels):
        """
        Initializes the BiLSTM-CRF model.

        Args:
            feature_dim (int): Dimension of input features per token (e.g., GloVe + TF-IDF + POS + NER).
            hidden_dim (int): Dimension of hidden states inside the LSTM (per direction).
            num_labels (int): Number of output labels (BIO tagging + propaganda types).
        """
        super(BiLSTM_CRF, self).__init__()

        # Bidirectional LSTM layer --> input feature vectors --> contextualized hidden states
        self.lstm = nn.LSTM(input_size=feature_dim,
                            hidden_size=hidden_dim,
                            bidirectional=True,
                            batch_first=True)

        # Linear layer: projects the BiLSTM hidden states to label space (logits for each tag)
        self.linear = nn.Linear(hidden_dim * 2, num_labels)  # *2 because it's bidirectional

        # CRF layer - models the dependencies between output labels and enforces valid sequences
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, x, mask, labels=None):

        #  Pass input features through the BiLSTM
        lstm_out, _ = self.lstm(x)  # Output shape: (batch_size, seq_len, hidden_dim * 2)

        #  Pass the LSTM outputs through a linear layer to get emission scores
        emissions = self.linear(lstm_out)  # Output shape: (batch_size, seq_len, num_labels)

        #  If labels are provided (training mode), compute the CRF loss
        if labels is not None:
            # CRF returns the log likelihood; we take the negative to minimize the loss
            loss = -self.crf(emissions, labels, mask=mask, reduction="token_mean")  # none|sum|mean|token_mean
            return loss

        # If no labels (inference mode), decode the best sequence of labels
        else:
            predictions = self.crf.decode(emissions, mask=mask)
            return predictions

In [ ]:
class BiLSTM_CRF2(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_labels, o_label_id=0, penalty_weight=0.05):
        """
        Initializes the BiLSTM-CRF model.
        This version Givrs penalty for using too many 'O' labels

        Args:
            feature_dim (int): Dimension of input features per token (e.g., GloVe + TF-IDF + POS + NER).
            hidden_dim (int): Dimension of hidden states inside the LSTM (per direction).
            num_labels (int): Number of output labels (BIO tagging + propaganda types).
            o_label_id (int): Label ID corresponding to "O" (default is 0).
            penalty_weight (float): Weight of the penalty for overpredicting "O".
        """
        super(BiLSTM_CRF2, self).__init__()

        self.lstm = nn.LSTM(input_size=feature_dim,
                            hidden_size=hidden_dim,
                            bidirectional=True,
                            batch_first=True)

        self.linear = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

        # Custom parameters for post-CRF reweighting
        self.o_label_id = o_label_id
        self.penalty_weight = penalty_weight

    def forward(self, x, mask, labels=None):

        # BiLSTM encoding
        lstm_out, _ = self.lstm(x)

        # Linear projection to emissions
        emissions = self.linear(lstm_out)

        # Training or inference
        if labels is not None:
            # Standard CRF negative log-likelihood
            loss = -self.crf(emissions, labels, mask=mask, reduction='token_mean')

            # ---- Post-CRF Reweighting ----
            if self.training:
                with torch.no_grad():
                    preds = self.crf.decode(emissions, mask=mask)
                    # preds is a list of list — convert to tensor
                    preds_tensor = torch.zeros_like(labels)
                    for i, seq in enumerate(preds):
                        preds_tensor[i, :len(seq)] = torch.tensor(seq, device=labels.device)

                    # Calculate percentage of 'O' predictions
                    percent_O = (preds_tensor == self.o_label_id).float().mean()

                # Add penalty proportional to % of "O" predictions
                loss += self.penalty_weight * percent_O
            return loss

        else:
            # Inference mode
            predictions = self.crf.decode(emissions, mask=mask)
            return predictions

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from seqeval.metrics import classification_report, f1_score
from torch.optim import AdamW

def decode_label_ids(label_ids, id2label):
    # Decode from label id to actual label
    return [[id2label[i] for i in sent] for sent in label_ids]

def evaluate(model, val_loader, id2label, device):
    """
    Evaluate model generated sequence of labels using seqeval
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            x = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)

            predictions = model(x, mask)  # list of lists of label IDs
            print(f"Prediction: {predictions[0]}\nTrue Labels: {labels[0]}")
            all_preds.extend(predictions)

            # Truncate labels to the actual sequence lengths (mask-based)
            labels = labels.cpu().numpy()
            masks = mask.cpu().numpy()

            for i in range(len(labels)):
                length = masks[i].sum()
                all_labels.append(labels[i][:length].tolist())

    # Decode label IDs to tag strings
    pred_tags = decode_label_ids(all_preds, id2label)
    true_tags = decode_label_ids(all_labels, id2label)

    print(classification_report(true_tags, pred_tags, digits=4))
    print("Validation F1 Score:", f1_score(true_tags, pred_tags))
    return f1_score(true_tags, pred_tags)


### Train the model

In [ ]:
def train_model(model, first_loader, train_loader, val_loader, id2label, num_epochs=10):
    """
    Train model
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    model.to(device)

    # SGD optimizer as per the paper
    #optimizer = optim.SGD(model.parameters(), lr=0.02)
    optimizer = AdamW(model.parameters(), lr=0.00005)

    best_f1 = 0.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        current_loader = first_loader if epoch < 5 else train_loader

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        for batch in tqdm(current_loader, desc="Training"):
            x = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)

            optimizer.zero_grad()
            loss = model(x, mask, labels)
            loss.backward()

            # Clip gradients to avoid exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(current_loader)
        print(f"Average Training Loss: {avg_loss:.4f}")

        # Evaluate on validation set
        f1 = evaluate(model, val_loader, id2label, device)

    print(f"\nTraining complete. Best F1: {best_f1:.4f}")


In [ ]:
# model without penalty
model = BiLSTM_CRF(feature_dim=335, hidden_dim=100, num_labels=len(label2id))

# model with penalty
model2 = BiLSTM_CRF2(feature_dim=335, hidden_dim=100, num_labels=len(label2id), penalty_weight=60)

# train
train_model(model2, train_loader, train_loader, val_loader, id2label, num_epochs=10)

cuda

Epoch 1/10


Training: 100%|██████████| 100/100 [02:29<00:00,  1.50s/it]


Average Training Loss: 62.1304
Prediction: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
True Labels: tensor([ 0,  0,  2, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
      

Training: 100%|██████████| 100/100 [02:29<00:00,  1.49s/it]


Average Training Loss: 62.2875
Prediction: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
True Labels: tensor([ 0,  0,  2, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
      

Training: 100%|██████████| 100/100 [02:28<00:00,  1.48s/it]


Average Training Loss: 61.6662
Prediction: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
True Labels: tensor([ 0,  0,  2, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
      

Training:  27%|██▋       | 27/100 [00:41<01:51,  1.53s/it]


KeyboardInterrupt: 